# Hinglish emotion training on Colab

Run exactly one GPU job per session. Select one JOB below. The default `xlmr_combined` trains one seven-emotion classifier on hi-en and fr-en. This is supervised task fine-tuning, not training a language model from scratch.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import files, drive
uploaded=files.upload()
zips=[n for n in uploaded if n.lower().endswith(".zip")]
assert len(zips)==1, "Upload exactly one generated training ZIP"
base=Path("/content").resolve()
with zipfile.ZipFile(zips[0]) as z:
    assert sum(i.file_size for i in z.infolist()) < 512*1024**2
    for i in z.infolist():
        assert "\\" not in i.filename and (base/i.filename).resolve().is_relative_to(base)
        assert (i.external_attr>>16)&0o170000 != 0o120000
    z.extractall(base)
PACKAGE=base/"hinglish_project_training"
m=json.loads((PACKAGE/"MANIFEST.json").read_text())
for name,digest in m["sha256"].items():
    p=(PACKAGE/name).resolve(); assert p.is_relative_to(PACKAGE)
    assert hashlib.sha256(p.read_bytes()).hexdigest()==digest, name
PROJECT=PACKAGE/"source"; os.chdir(PROJECT)
subprocess.run([sys.executable,"-m","pip","install","-e",".[model]"],check=True)
import torch
assert torch.cuda.is_available(), "Enable Runtime > Change runtime type > T4 GPU, then restart from this cell"
print("GPU:",torch.cuda.get_device_name(0))


In [ ]:
# Select exactly one job for this Colab session.
JOB="xlmr_combined"  # xlmr_combined | mbert_combined | muril_hi_en | sarcasm_hi_en
EPOCHS=3
USE_OWN_EMOTION_DATA=False
assert JOB in {"xlmr_combined","mbert_combined","muril_hi_en","sarcasm_hi_en"}
drive.mount("/content/drive")
STORAGE=Path("/content/drive/MyDrive/HinglishEmotionProject")
STORAGE.mkdir(parents=True,exist_ok=True)
DATA=PACKAGE/"data"/"all_splits.csv"
AUDIT=PACKAGE/"data"/"dataset_audit.json"
if USE_OWN_EMOTION_DATA:
    up=files.upload(); csvs=[n for n in up if n.lower().endswith(".csv")]
    assert len(csvs)==1, "Upload one reviewed canonical emotion CSV"
    DATA=Path(csvs[0]).resolve()
    AUDIT=None


## Dataset and models

Bootstrap: synthetic, not human-reviewed; it cannot establish real-world emotion accuracy. The frozen splits are copied to Drive and reused on resume. For reviewed emotion data, supply `text,emotion,language_pair` and keep group identities within a single explicit split when supplied. Do not transform binary sentiment or emotion labels into sarcasm labels. Model cards: [XLM-R](https://huggingface.co/FacebookAI/xlm-roberta-base), [mBERT](https://huggingface.co/google-bert/bert-base-multilingual-cased), [MuRIL](https://huggingface.co/google/muril-base-cased). MuRIL job is Hindi-English only.

For `sarcasm_hi_en`, upload your own independently human-labelled CSV with `text,sarcasm,language_pair,split,group_id,human_reviewed`. Labels must be `sarcastic` or `not_sarcastic` (0/1 accepted). Both classes must occur in each split; group ids and exact duplicate texts may not cross splits. Synthetic rows are refused unless explicitly opted in in the trainer CLI. The package includes a header-only template and no sarcasm examples.


In [ ]:
if JOB=="sarcasm_hi_en":
    up=files.upload(); csvs=[n for n in up if n.lower().endswith(".csv")]
    assert len(csvs)==1, "Upload your human-labelled sarcasm CSV; no fallback dataset exists"
    command=[sys.executable,"-u","-m","training.train_sarcasm","--data",str(Path(csvs[0]).resolve()),"--model","FacebookAI/xlm-roberta-base","--output",str(STORAGE/"sarcasm_hi_en"),"--epochs",str(EPOCHS),"--resume"]
else:
    configs={"xlmr_combined":("xlmr","all"),"mbert_combined":("mbert","all"),"muril_hi_en":("muril","hi-en")}
    model,language=configs[JOB]
    run=STORAGE/JOB; run.mkdir(parents=True,exist_ok=True)
    from training.data_workflow import prepare_data
    prepared=prepare_data(DATA,"emotion","raw",42,run/"frozen_split.csv",AUDIT if AUDIT and AUDIT.exists() else None,language,allow_synthetic=not USE_OWN_EMOTION_DATA)
    command=[sys.executable,"-u","-m","training.compare_encoders","--task","emotion","--models",model,"--data",str(DATA),"--language-pair",language,"--output-root",str(run/"models"),"--split-manifest",str(run/"frozen_split.csv"),"--epochs",str(EPOCHS),"--batch-size","2","--gradient-accumulation","8","--max-length","128","--learning-rate","2e-5","--precision","auto","--text-mode","raw","--seed","42","--split-seed","42","--resume"]
    if AUDIT and AUDIT.exists(): command += ["--data-audit",str(AUDIT)]
    if not USE_OWN_EMOTION_DATA: command += ["--allow-synthetic"]
subprocess.run(command,cwd=PROJECT,check=True)


## Results

Seven emotion checkpoints report validation selection and held-out test results. Sarcasm is a separate binary classifier and will not be merged with the emotion output. Save the full Drive folder to resume. Inference exports should include only tokenizer/config and safetensors; exclude raw rows and optimizer state.


In [ ]:
# Inspect held-out results and export the best inference checkpoint.
from training.cloud_bundle import export_checkpoint, export_sarcasm_checkpoint
if JOB=="sarcasm_hi_en":
    model_dir=STORAGE/"sarcasm_hi_en"
    metadata=json.loads((model_dir/"sarcasm_metadata.json").read_text())
    print("Task:",metadata.get("task"),"Base model:",metadata.get("base_model"))
    print("Selected validation macro-F1:",metadata.get("selected_validation_macro_f1"))
    display(metadata.get("test_metrics",{}))
    export_path=STORAGE/"exports"/"sarcasm_hi_en.zip"
    export_sarcasm_checkpoint(model_dir,export_path)
else:
    model_dir=run/"models"/(model+"_emotion")
    metadata=json.loads((model_dir/"emotion_metadata.json").read_text())
    metrics=json.loads((model_dir/"metrics.json").read_text())
    print("Task:",metadata.get("task"),"Base model:",metadata.get("base_model"))
    print("Selected validation macro-F1:",metrics.get("best_validation_macro_f1"))
    display(metrics.get("test",{}))
    export_path=STORAGE/"exports"/(JOB+".zip")
    export_checkpoint(model_dir,export_path)
print("Export:",export_path)
files.download(str(export_path))
